In [1]:
import importlib
import json
import random
import sys
import time
from pathlib import Path
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

REPO_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "data/sft/dev.jsonl").is_file()
     and (candidate / "src/sellerops/dataset.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not locate sellerops-agent root from the notebook working directory"
    )

sys.path.insert(0, str(REPO_ROOT / "src"))

# A notebook kernel may cache an older local package after the dataset is regenerated.
# Remove cached sellerops modules so the source under this repo is imported.
for module_name in list(sys.modules):
    if module_name == "sellerops" or module_name.startswith("sellerops."):
        del sys.modules[module_name]

import sellerops.schemas as sellerops_schemas
import sellerops.dataset as sellerops_dataset
importlib.reload(sellerops_schemas)
importlib.reload(sellerops_dataset)
DATASET_VERSION = sellerops_dataset.DATASET_VERSION
SYSTEM_PROMPT = sellerops_dataset.SYSTEM_PROMPT
SchemaError = sellerops_schemas.SchemaError
canonical_json = sellerops_schemas.canonical_json
validate_action = sellerops_schemas.validate_action

MODEL_ID = "Qwen/Qwen3-0.6B-Base"
SEED = 20260820
MAX_NEW_TOKENS = 256

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

/Users/emeryzwageri/miniforge3/envs/llm-post-training/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_REVISION = "da87bfb608c14b7cf20ba1ce41287e8de496c0cd"
print(MODEL_REVISION)

da87bfb608c14b7cf20ba1ce41287e8de496c0cd


In [3]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DTYPE = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    DTYPE = torch.float16
else:
    DEVICE = torch.device("cpu")
    DTYPE = torch.float32

print("device:", DEVICE)
print("dtype:", DTYPE)

device: mps
dtype: torch.float16


In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=DTYPE,
    low_cpu_mem_usage=True,
)

model.to(DEVICE)
model.eval()

print(type(model))
print(tokenizer.special_tokens_map)
print(tokenizer.chat_template[:1000])

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 345.64it/s]


<class 'transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM'>
{'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}
{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- set ns = namespace(multi_step

In [5]:
dev_records = load_dataset(
    "json",
    data_files={"dev": str(REPO_ROOT / "data/sft/dev.jsonl")},
    split="dev",
)

assert set(dev_records["dataset_version"]) == {DATASET_VERSION}
assert dev_records[0]["messages"][0]["content"] == SYSTEM_PROMPT

print(dev_records)
print(dev_records[0])

Dataset({
    features: ['dataset_version', 'environment_version', 'family', 'locale', 'messages', 'provenance', 'record_id', 'split', 'target_message_index', 'task_id', 'template_id'],
    num_rows: 78
})
{'dataset_version': 'sellerops-sft-v0.2', 'environment_version': 'sellerops-sim-v0.1', 'family': 'product_search', 'locale': 'id', 'messages': [{'content': 'You are the SellerOps decision policy.\n\nReturn exactly one valid JSON object and nothing else. Do not return Markdown,\ncode fences, explanations, reasoning, or multiple JSON values.\n\nThe available actions and their exact schemas are:\n\n1. Call one tool:\n{"type":"tool_call","tool":"<tool_name>","arguments":{...}}\n\nAvailable tools:\n- search_products: Find products by query and optional filters.\n  Required arguments: query (string) and filters (object). Filters may contain\n  only brand (string), category (string), color (string), in_stock (boolean),\n  and max_price_idr (integer).\n- get_inventory: Retrieve current inven

In [6]:
record = dev_records[0]

input_messages = record["messages"][:-1]
expected_action = json.loads(record["messages"][-1]["content"])

In [7]:
rendered = tokenizer.apply_chat_template(
    input_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(rendered)

<|im_start|>system
You are the SellerOps decision policy.

Return exactly one valid JSON object and nothing else. Do not return Markdown,
code fences, explanations, reasoning, or multiple JSON values.

The available actions and their exact schemas are:

1. Call one tool:
{"type":"tool_call","tool":"<tool_name>","arguments":{...}}

Available tools:
- search_products: Find products by query and optional filters.
  Required arguments: query (string) and filters (object). Filters may contain
  only brand (string), category (string), color (string), in_stock (boolean),
  and max_price_idr (integer).
- get_inventory: Retrieve current inventory for one known product.
  Required arguments: product_id (string).
- get_discount_policy: Retrieve the discount policy for one known product.
  Required arguments: product_id (string).
- calculate_offer: Calculate a policy-compliant offer for one known product.
  Required arguments: product_id (string), quantity (integer of at least 1),
  and discount_p

In [8]:
tokenizer.padding_side = "left"

In [9]:
import torch

@torch.inference_mode()
def generate_actions(
    messages_batch: list[list[dict]],
) -> list[str]:

    pad_token_id = tokenizer.pad_token_id

    if pad_token_id is None:
        pad_token_id = tokenizer.eos_token_id

    inputs = tokenizer.apply_chat_template(
        messages_batch,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        padding=True,
        return_tensors="pt",
        return_dict=True,
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    eos_token_ids = []
    for token_id in (
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|im_end|>"),
    ):
        if token_id is not None and token_id not in eos_token_ids:
            eos_token_ids.append(token_id)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=pad_token_id,
        eos_token_id=eos_token_ids,
    )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = generated_ids[:, input_length:]

    return tokenizer.batch_decode(
        new_tokens,
        skip_special_tokens=True,
    )

In [10]:
raw_prediction = generate_actions([input_messages])[0].strip()

print("OUTPUT MODEL:")
print(raw_prediction)

print("\nEXPECTED:")
print(canonical_json(expected_action))

OUTPUT MODEL:
คำตอบ: {"type":"tool_call","tool":"search_products","arguments":{"query":"Comet","filters":{"brand":"Comet"}}}

EXPECTED:
{"arguments":{"filters":{"brand":"Comet","in_stock":true},"query":"backpack"},"tool":"search_products","type":"tool_call"}


## Strict output parsing

The primary baseline metric accepts only one valid JSON object that satisfies the SellerOps schema. Prefixes, Markdown fences, native tool wrappers, and repaired JSON remain failures. This prevents post-processing from hiding model errors.

Parser berikut membedakan JSON yang rusak dari JSON valid dengan schema SellerOps yang salah.

In [11]:
def parse_action(raw_text: str) -> dict:
    text = raw_text.strip()

    try:
        action = json.loads(text)
    except json.JSONDecodeError as error:
        return {
            "action": None,
            "json_valid": False,
            "schema_valid": False,
            "error_type": "json_decode_error",
            "error": str(error),
        }

    try:
        validate_action(action)
    except SchemaError as error:
        return {
            "action": action,
            "json_valid": True,
            "schema_valid": False,
            "error_type": "schema_error",
            "error": str(error),
        }

    return {
        "action": action,
        "json_valid": True,
        "schema_valid": True,
        "error_type": None,
        "error": None,
    }

In [12]:
valid_check = parse_action(canonical_json(expected_action))
assert valid_check["json_valid"] is True
assert valid_check["schema_valid"] is True
assert valid_check["action"] == expected_action

prefixed_check = parse_action("answer: " + canonical_json(expected_action))
assert prefixed_check["json_valid"] is False
assert prefixed_check["schema_valid"] is False

wrong_schema = canonical_json({
    "type": "tool_call",
    "tool": "made_up_tool",
    "arguments": {},
})
wrong_schema_check = parse_action(wrong_schema)
assert wrong_schema_check["json_valid"] is True
assert wrong_schema_check["schema_valid"] is False

print("parser self-check passed")

parsed_prediction = parse_action(raw_prediction)
exact_match = (
    parsed_prediction["schema_valid"]
    and parsed_prediction["action"] == expected_action
)

print(json.dumps(parsed_prediction, ensure_ascii=False, indent=2))
print("exact_match:", exact_match)

parser self-check passed
{
  "action": null,
  "json_valid": false,
  "schema_valid": false,
  "error_type": "json_decode_error",
  "error": "Expecting value: line 1 column 1 (char 0)"
}
exact_match: False


## Development-set evaluation

Run only on `dev` while the prompt, parser, and metrics are still being debugged. The frozen test tasks remain untouched until this evaluator is fixed.

Evaluator membandingkan action secara bertingkat. Tool dan argument metrics bernilai `None` jika target bukan `tool_call`, sehingga record tersebut tidak masuk denominator kedua metrik itu.

In [13]:
def evaluate_prediction(record: dict, raw_prediction: str) -> dict:
    expected = json.loads(record["messages"][-1]["content"])
    parsed = parse_action(raw_prediction)
    predicted = parsed["action"]
    predicted_is_object = isinstance(predicted, dict)
    expected_is_tool_call = expected["type"] == "tool_call"

    action_type_correct = (
        predicted_is_object
        and predicted.get("type") == expected["type"]
    )
    tool_choice_correct = (
        predicted_is_object
        and predicted.get("tool") == expected["tool"]
    ) if expected_is_tool_call else None
    arguments_exact = (
        predicted_is_object
        and predicted.get("tool") == expected["tool"]
        and predicted.get("arguments") == expected["arguments"]
    ) if expected_is_tool_call else None
    exact_action_match = (
        parsed["schema_valid"]
        and predicted == expected
    )

    return {
        "record_id": record["record_id"],
        "task_id": record["task_id"],
        "family": record["family"],
        "locale": record["locale"],
        "raw_prediction": raw_prediction.strip(),
        "predicted_action": predicted,
        "expected_action": expected,
        "json_valid": parsed["json_valid"],
        "schema_valid": parsed["schema_valid"],
        "action_type_correct": action_type_correct,
        "tool_choice_correct": tool_choice_correct,
        "arguments_exact": arguments_exact,
        "exact_action_match": exact_action_match,
        "error_type": parsed["error_type"],
        "error": parsed["error"],
    }

In [14]:
perfect_result = evaluate_prediction(record, canonical_json(expected_action))
assert perfect_result["action_type_correct"] is True
assert perfect_result["tool_choice_correct"] is True
assert perfect_result["arguments_exact"] is True
assert perfect_result["exact_action_match"] is True

wrong_arguments = dict(expected_action)
wrong_arguments["arguments"] = {}
wrong_arguments_result = evaluate_prediction(
    record,
    canonical_json(wrong_arguments),
)
assert wrong_arguments_result["action_type_correct"] is True
assert wrong_arguments_result["tool_choice_correct"] is True
assert wrong_arguments_result["arguments_exact"] is False
assert wrong_arguments_result["exact_action_match"] is False

final_record = next(
    row
    for row in dev_records
    if json.loads(row["messages"][-1]["content"])["type"] == "final"
)
final_expected = json.loads(final_record["messages"][-1]["content"])
final_result = evaluate_prediction(final_record, canonical_json(final_expected))
assert final_result["tool_choice_correct"] is None
assert final_result["arguments_exact"] is None
assert final_result["exact_action_match"] is True

print("evaluation self-check passed")

evaluation self-check passed


In [17]:
BATCH_SIZE = 4  # Reduce to 2 if MPS runs out of memory.
dev_results = []
evaluation_started = time.perf_counter()

for start in range(0, 4, BATCH_SIZE):
    stop = min(start + BATCH_SIZE, len(dev_records))
    batch_records = [dev_records[index] for index in range(start, stop)]
    messages_batch = [row["messages"][:-1] for row in batch_records]

    batch_started = time.perf_counter()
    batch_predictions = generate_actions(messages_batch)
    batch_seconds = time.perf_counter() - batch_started

    for row, prediction in zip(batch_records, batch_predictions, strict=True):
        result = evaluate_prediction(row, prediction)
        result["approx_generation_seconds"] = batch_seconds / len(batch_records)
        dev_results.append(result)

    print(f"evaluated {stop:>3}/{len(dev_records)} records")

evaluation_seconds = time.perf_counter() - evaluation_started
# assert len(dev_results) == len(dev_records)
print(f"completed in {evaluation_seconds:.1f} seconds")

evaluated   4/78 records
completed in 46.0 seconds


In [18]:
# Nilai None dikeluarkan dari denominator karena metric tidak applicable.
def summarize_boolean(rows: list[dict], field: str) -> dict:
    eligible = [row for row in rows if row[field] is not None]
    correct = sum(bool(row[field]) for row in eligible)
    total = len(eligible)
    return {
        "correct": correct,
        "total": total,
        "rate": correct / total if total else None,
    }

metric_self_check = summarize_boolean(
    [{"score": True}, {"score": False}, {"score": None}],
    "score",
)
assert metric_self_check == {"correct": 1, "total": 2, "rate": 0.5}
print("metric self-check passed")

metric_fields = [
    "json_valid",
    "schema_valid",
    "action_type_correct",
    "tool_choice_correct",
    "arguments_exact",
    "exact_action_match",
]

metrics = {
    field: summarize_boolean(dev_results, field)
    for field in metric_fields
}

summary = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset_version": DATASET_VERSION,
    "split": "dev",
    "record_count": len(dev_results),
    "seed": SEED,
    "device": str(DEVICE),
    "dtype": str(DTYPE),
    "generation": {
        "do_sample": False,
        "max_new_tokens": MAX_NEW_TOKENS,
        "batch_size": BATCH_SIZE,
        "enable_thinking": False,
    },
    "evaluation_seconds": evaluation_seconds,
    "metrics": metrics,
}

print(json.dumps(summary, ensure_ascii=False, indent=2))

metric self-check passed
{
  "model_id": "Qwen/Qwen3-0.6B-Base",
  "model_revision": "da87bfb608c14b7cf20ba1ce41287e8de496c0cd",
  "dataset_version": "sellerops-sft-v0.2",
  "split": "dev",
  "record_count": 4,
  "seed": 20260820,
  "device": "mps",
  "dtype": "torch.float16",
  "generation": {
    "do_sample": false,
    "max_new_tokens": 256,
    "batch_size": 4,
    "enable_thinking": false
  },
  "evaluation_seconds": 45.983970959001454,
  "metrics": {
    "json_valid": {
      "correct": 0,
      "total": 4,
      "rate": 0.0
    },
    "schema_valid": {
      "correct": 0,
      "total": 4,
      "rate": 0.0
    },
    "action_type_correct": {
      "correct": 0,
      "total": 4,
      "rate": 0.0
    },
    "tool_choice_correct": {
      "correct": 0,
      "total": 2,
      "rate": 0.0
    },
    "arguments_exact": {
      "correct": 0,
      "total": 2,
      "rate": 0.0
    },
    "exact_action_match": {
      "correct": 0,
      "total": 4,
      "rate": 0.0
    }
  }
}


In [19]:
failures = [row for row in dev_results if not row["exact_action_match"]]
print(f"failures: {len(failures)}/{len(dev_results)}")

for failure in failures[:5]:
    print("\n" + "=" * 80)
    print("record_id:", failure["record_id"])
    print("error:", failure["error_type"], failure["error"])
    print("prediction:", failure["raw_prediction"])
    print("expected:", canonical_json(failure["expected_action"]))

failures: 4/4

record_id: dev-search-dev-brand-id-000-a00
error: json_decode_error Expecting value: line 1 column 1 (char 0)
prediction: คำตอบ: {"type":"tool_call","tool":"search_products","arguments":{"query":"Comet","filters":{"brand":"Comet"}}}
expected: {"arguments":{"filters":{"brand":"Comet","in_stock":true},"query":"backpack"},"tool":"search_products","type":"tool_call"}

record_id: dev-search-dev-brand-id-000-a01
error: json_decode_error Extra data: line 2 column 1 (char 159)
prediction: {"error":null,"ok":true,"result":{"products":[{"name":"Comet Everyday Backpack","price_idr":499000,"product_id":"P1008","stock":5}]},"tool":"search_products"}
[]{"error":null,"ok":true,"result":{"products":[{"name":"Comet Everyday Backpack","price_idr":499000,"product_id":"P1008","stock":5}]},"tool":"search_products"}
[]{"error":null,"ok":true,"result":{"products":[{"name":"Comet Everyday Backpack","price_idr":499000,"product_id":"P1008","stock":5}]},"tool":"search_products"}
[]{"error":null,"o

## Save the reproducible development baseline

Each execution writes a timestamped result pair so an earlier run is not overwritten. Raw predictions are retained for error analysis.

In [20]:
results_dir = REPO_ROOT / "results" / "baseline"
results_dir.mkdir(parents=True, exist_ok=True)

run_timestamp = time.strftime("%Y%m%d-%H%M%S")
model_slug = MODEL_ID.split("/")[-1].lower()
run_name = (
    f"{run_timestamp}__{model_slug}__{DATASET_VERSION}__dev"
)
raw_results_path = results_dir / f"{run_name}.jsonl"
summary_path = results_dir / f"{run_name}.summary.json"

with raw_results_path.open("x", encoding="utf-8") as file:
    for result in dev_results:
        file.write(json.dumps(result, ensure_ascii=False, sort_keys=True) + "\n")

with summary_path.open("x", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=2, sort_keys=True)
    file.write("\n")

print(raw_results_path)
print(summary_path)

/Users/emeryzwageri/llm-post-training-lab/projects/sellerops-agent/results/baseline/20260822-190247__qwen3-0.6b-base__sellerops-sft-v0.2__dev.jsonl
/Users/emeryzwageri/llm-post-training-lab/projects/sellerops-agent/results/baseline/20260822-190247__qwen3-0.6b-base__sellerops-sft-v0.2__dev.summary.json


## Stop here before frozen test

Review at least five development failures and freeze the parser, generation settings, and metric definitions first. Do not tune the prompt using frozen test tasks. After the evaluator is stable, build the separate frozen-task evaluation path and run it once before SFT.